In [ ]:
import nglview as nv
import mdtraj as md

import configparser
import os

# 1. Initialize Config Parser
config = configparser.ConfigParser()
config.read('path_cfg.cfg')

#  Extract paths
base = os.getenv("MDSIM_DATASET_PATH") 
xtc_trajectory = "chignolin_trajectories\filtered\e118s2_e53s6p0f119\e1s1_chignolin_50ns_0-ADRIA_CHIG_ADAPTIVE_crystal_ss_contacts_50_chignolin_0-0-1-RND3469_9.filtered.xtc"
pdb_file = "chignolin_trajectories\filtered\filtered.pdb"
psf_file = "chignolin_trajectories\filtered\filtered.psf"

pdb_path = os.path.join(base, pdb_file)
traj_path = os.path.join(base, xtc_trajectory)


t = md.load(traj_path, top=pdb_path)

# View it
view = nv.show_mdtraj(t)
view

view.clear_representations()
view.add_representation('cartoon', selection='protein', color='residueindex')
view.add_representation('ball+stick', selection='protein')
view.center('protein')

# --- REPLACING THE BROKEN PLAYER CODE ---
# In newer nglview, we use the underscore player or handle via the widget directly
if hasattr(view, 'player'):
    view.player.parameters = dict(delay=50, step=1)
elif hasattr(view, '_player'):
    view._player.parameters = dict(delay=50, step=1)

# To force the player to show up, we set this:
view.display(gui=True) 
# ----------------------------------------

view


NGLWidget(gui_style='ngl', max_frame=499)

In [ ]:
import mdtraj as md
import nglview as nv
import numpy as np
import os
import configparser

# 1. Setup Paths
config = configparser.ConfigParser()
config.read('path_cfg.cfg')
base = config['PATHS']['base_folder']
xtc_sub = config['PATHS']['xtc_subfolder']

#  Extract paths
base = os.getenv("MDSIM_DATASET_PATH") 
xtc_trajectory = "chignolin_trajectories\filtered\e118s2_e53s6p0f119\e1s1_chignolin_50ns_0-ADRIA_CHIG_ADAPTIVE_crystal_ss_contacts_50_chignolin_0-0-1-RND3469_9.filtered.xtc"
pdb_file = "chignolin_trajectories\filtered\filtered.pdb"
psf_file = "chignolin_trajectories\filtered\filtered.psf"

pdb_path = os.path.join(base, pdb_file)
traj_path = os.path.join(base, xtc_trajectory)

# 2. Load Trajectory
t = md.load(traj_path, top=pdb_path)

# 3. Apply the "Make Whole" Fix
# We force the bond graph and manually set the anchor to bypass the PSF errors
t.topology.create_standard_bonds()
molecules = t.topology.find_molecules()
t.image_molecules(inplace=True, anchor_molecules=[molecules[0]])

# 4. Center the Protein at (0, 0, 0)
# Essential for the JAX-MD neighbor search grid
t.center_coordinates()

# 5. Coordinate Extraction & Stats
# Convert nm to Angstroms
coords_gnn = t.xyz * 10.0 

# Calculate the max distance between any two atoms in the first frame
# (This should be ~37.88 Å based on your last run)
frame_0 = coords_gnn[0]
dist_matrix = np.linalg.norm(frame_0[:, None, :] - frame_0[None, :, :], axis=-1)
max_d = np.max(dist_matrix)

print(f"--- Coordinate Validation ---")
print(f"Trajectory processed: {len(t)} frames")
print(f"Final Max Diameter (Frame 0): {max_d:.4f} Å")
print("-" * 30)

# 6. Visualize (Your preferred NGLView setup)
view = nv.show_mdtraj(t)
view.clear_representations()
view.add_representation('cartoon', selection='protein', color='residueindex')
view.add_representation('ball+stick', selection='protein')
view.center('protein')

# Player settings for newer nglview versions
if hasattr(view, 'player'):
    view.player.parameters = dict(delay=50, step=1)
elif hasattr(view, '_player'):
    view._player.parameters = dict(delay=50, step=1)

view.display(gui=True)
view

--- Coordinate Validation ---
Trajectory processed: 500 frames
Final Max Diameter (Frame 0): 38.8564 Å
------------------------------


NGLWidget(gui_style='ngl', max_frame=499)

In [59]:
import time

# Verify the connection by jumping through frames
print(len(t))
for frame in range(0, len(t), 1):
    view.frame = frame
    time.sleep(0.1) # Small delay to see it move

500


KeyboardInterrupt: 